<a href="https://colab.research.google.com/github/VictorNikolai/Proyecto_Tesis-/blob/main/Entrenar_Modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenar U-Net y DeepLabV3+ con Landslide4Sense, exportarlos y verificarlos

Este cuaderno deja los dos modelos listos para el sistema, con las entradas que pide
el proyecto:

| Canal | Qué es | Unidades en Landslide4Sense |
|---|---|---|
| B1, B2, B3, B4 | Sentinel-2: aerosol, azul, verde, rojo | reflectancia dividida por su media |
| B13 | pendiente (ALOS PALSAR) | grados / 20.292 |
| B14 | DEM (ALOS PALSAR) | metros / 772.82 |

Lo que lo hace correcto esta vez:

1. **El tensor se arma con la misma función del servidor** (`recetas.desde_l4s`).
   Entrenar y detectar normalizan igual por construcción.
2. **Normalización media/desviación calculada sobre el entrenamiento**, que viaja en
   la receta junto al ONNX.
3. **Partición oficial**: entrenamiento (3799), validación (245) para elegir la época
   y el umbral, y prueba (800) medida una sola vez al final.
4. **Exportación con carga estricta, paridad con PyTorch y revisión de pesos**: un
   ONNX sin entrenar no sale de aquí.
5. **Verificación del ONNX** con la receta del servidor: tiene que dar las mismas
   métricas que el entrenamiento.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.
Tiempo aproximado: 10 min de descarga y unos 30 a 45 min por modelo.

In [1]:
!pip -q install segmentation-models-pytorch onnx onnxruntime h5py huggingface_hub

## Configuración

In [2]:
MODELOS = {                       # arquitectura -> autor (se guarda en la receta)
    'unet': 'Mejía López, L. C.',
    'deeplabv3plus': 'Huarcaya Pumacayo, V. N.',
}
CANALES = 'B01,B02,B03,B04,B13,B14'   # Sentinel-2 B1 a B4, pendiente, DEM
EPOCAS = 100                      # como la Fase 4; con coseno conviene recorrerlas todas
PACIENCIA = 30
CARPETA_DRIVE = '/content/drive/MyDrive/entrenamiento_b1b4b13b14'   # checkpoints y resultados
SALIDA_ONNX = '/content/drive/MyDrive/onnx_para_render'              # lo que va a backend/modelos/

In [3]:
import torch
assert torch.cuda.is_available(), 'activa la GPU: Entorno de ejecución > Cambiar tipo de entorno > T4'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(CARPETA_DRIVE, exist_ok=True); os.makedirs(SALIDA_ONNX, exist_ok=True)

GPU: Tesla T4
Mounted at /content/drive


## El código del proyecto

Copiado sin cambios de `backend/nucleo` y `herramientas`.

In [4]:
import os
ARCHIVOS = {}
ARCHIVOS['backend/nucleo/__init__.py'] = ''
ARCHIVOS['backend/nucleo/config.py'] = '"""\nTodo lo configurable del backend, en un solo sitio.\n\nCada valor se puede cambiar con una variable de entorno del mismo nombre, que es\ncomo se ajusta en Render (Environment) sin tocar el código.\n"""\nimport json\nimport os\nimport tempfile\n\n\ndef _env(nombre, defecto):\n    return os.environ.get(nombre, defecto)\n\n\nRAIZ = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\n\n# --------------------------------------------------------------------- modelos\n# Cada modelo es un .onnx más una receta .json con el mismo nombre de base, que\n# dice qué canales recibe, cómo se normalizan, qué salida tiene y su umbral.\nCARPETA_MODELOS = _env(\'CARPETA_MODELOS\', os.path.join(RAIZ, \'modelos\'))\n# Si un .onnx llega como puntero de Git LFS, se baja de aquí: <URL>/<archivo>\nURL_PESOS_BASE = _env(\'URL_PESOS_BASE\', \'\')\n# Un ONNX con pesos de inicialización no detecta nada aunque corra. Se rechaza\n# salvo que esto valga 1, que solo sirve para probar el circuito.\nPERMITIR_SIN_ENTRENAR = _env(\'PERMITIR_SIN_ENTRENAR\', \'0\') == \'1\'\nHILOS_ONNX = int(_env(\'HILOS_ONNX\', \'1\'))\n# Optimización del grafo: \'extendida\' da los mismos resultados (diferencia < 1e-6)\n# que \'todas\' y usa unos 100 MB menos, porque no duplica los pesos en el formato\n# interno NCHWc. En 512 MB eso decide si el servicio cabe o no.\nNIVEL_OPT_ONNX = _env(\'NIVEL_OPT_ONNX\', \'extendida\')\n# Hilos para bajar las imágenes. dask usa por defecto uno por núcleo de la\n# máquina, y en Render la máquina física tiene muchos aunque el plan dé 0.1 CPU:\n# cada hilo abre sus propias conexiones y cachés. Con 4 la descarga es rápida\n# (la espera es de red, no de cálculo) y la memoria queda acotada.\nHILOS_DESCARGA = int(_env(\'HILOS_DESCARGA\', \'4\'))\n# En el plan gratuito de Render hay 512 MB: mejor un modelo cargado a la vez.\nUN_MODELO_A_LA_VEZ = _env(\'UN_MODELO_A_LA_VEZ\', \'1\') == \'1\'\n# La sesión se carga recién para inferir y se suelta al terminar cada trabajo, así\n# no ocupa memoria mientras se bajan las imágenes. Con más RAM se puede poner 0.\nSOLTAR_SESION = _env(\'SOLTAR_SESION\', \'1\') == \'1\'\n# Parches por llamada al modelo. 4 da la misma velocidad que 8 en CPU y ~20 MB menos.\nLOTE_INFERENCIA = int(_env(\'LOTE_INFERENCIA\', \'4\'))\n# Al revisar cada modelo, el servidor guarda aquí una copia con los pesos en un\n# archivo aparte (.onnx.data). ONNX Runtime los lee directo del disco: unos 95 MB\n# dejan de contar como memoria propia del proceso y el resultado es idéntico.\n# Vacío para desactivarlo.\nCARPETA_COPIAS = _env(\'CARPETA_COPIAS\', os.path.join(\n    tempfile.gettempdir(), \'modelos_pesos_aparte\'))\n\n# ----------------------------------------------------------------------- CORS\n# La URL del Space, p. ej. https://usuario-espacio.hf.space. \'*\' para probar.\nORIGENES = [o.strip() for o in _env(\'ORIGENES_PERMITIDOS\', \'*\').split(\',\') if o.strip()]\n\n# ---------------------------------------------------------- fuente de imágenes\nSTAC = \'https://planetarycomputer.microsoft.com/api/stac/v1\'\nCOLECCION_S2 = \'sentinel-2-l2a\'\n# ----------------------------------------------------- relieve (B13 y B14)\n# Landslide4Sense toma la pendiente (B13) y el DEM (B14) del «ALOS PALSAR DEM» de\n# 12.5 m que distribuye ASF con los productos RTC. Ese DEM no sale de PALSAR: es\n# SRTM de 30 m remuestreado a 12.5 m (NASA Earthdata, proyecto ALOS PALSAR RTC).\n# Por eso la fuente por defecto, que cubo pide a Planetary Computer, es NASADEM\n# (SRTM reprocesado, 30 m). Otras opciones:\n#   alos-aw3d30   ALOS World 3D-30m de JAXA (óptico PRISM, misión ALOS)\n#   cop-dem-glo-30 Copernicus GLO-30\n#   asf-alos-palsar el mosaico de los .dem.tif de ASF para la provincia, armado\n#                  una vez con herramientas/alos_palsar_asf.py (archivo o URL COG)\n# Ninguna es una serie temporal: cada una va con su propia ventana de fechas.\nFUENTES_DEM = {\n    \'nasadem\': {\'coleccion\': \'nasadem\', \'banda\': \'elevation\',\n                \'fechas\': (\'2000-01-01\', \'2000-12-31\'),\n                \'nombre\': \'NASADEM (SRTM, 30 m), la base del DEM «ALOS PALSAR» de ASF\'},\n    \'alos-aw3d30\': {\'coleccion\': \'alos-dem\', \'banda\': \'data\',\n                    \'fechas\': (\'2016-01-01\', \'2016-12-31\'),\n                    \'nombre\': \'ALOS World 3D-30m (JAXA, 30 m)\'},\n    \'cop-dem-glo-30\': {\'coleccion\': \'cop-dem-glo-30\', \'banda\': \'data\',\n                       \'fechas\': (\'2020-01-01\', \'2023-12-31\'),\n                       \'nombre\': \'Copernicus DEM GLO-30 (30 m)\'},\n    \'asf-alos-palsar\': {\'archivo\': True,\n                        \'nombre\': \'ALOS PALSAR RTC (ASF), mosaico de la provincia\'},\n}\nFUENTE_DEM = _env(\'FUENTE_DEM\', \'nasadem\')\nif FUENTE_DEM not in FUENTES_DEM:\n    raise ValueError(f\'FUENTE_DEM="{FUENTE_DEM}" no es una de {sorted(FUENTES_DEM)}\')\n# Para asf-alos-palsar: ruta local o URL (https://...) del GeoTIFF del mosaico.\nRUTA_DEM_ASF = _env(\'RUTA_DEM_ASF\', \'\')\n# Constantes de Landslide4Sense, medidas en sus parches (son las mismas en todas\n# las regiones): pendiente = grados / 20.292 y elevación = metros / 772.82.\nESCALA_TERRENO_L4S = {\'pendiente\': 20.292, \'elevacion\': 772.82}\nRESOLUCION_M = 10\n# Lado de cada cubo de cubo. 256 px son 2.56 km: pocas llamadas y poca memoria.\nLADO_TESELA_PX = int(_env(\'LADO_TESELA_PX\', \'256\'))\n# Escenas por tesela, las menos nubosas según el metadato. Acota memoria y tiempo.\nMAX_ESCENAS = int(_env(\'MAX_ESCENAS\', \'6\'))\n# Clases SCL que se descartan: sin dato, saturado, sombra de nube, nube media,\n# nube alta y cirro.\nSCL_MALAS = (0, 1, 3, 8, 9, 10)\n# Desde la baseline 04.00 (enero de 2022) los niveles digitales de L2A llevan\n# sumado 1000. Se resta por escena.\nDESPLAZAMIENTO_L2A = 1000.0\n\n# ---------------------------------------------- unidades de Landslide4Sense\n# Landslide4Sense trae cada banda dividida por su media de escena. Estas son las\n# medias de La Convención (reflectancia L2A en niveles digitales), medidas con\n# herramientas/calibrar_escala.py sobre la provincia en temporada seca.\n_RUTA_ESCALA = _env(\'RUTA_ESCALA\', os.path.join(RAIZ, \'escala_laconvencion.json\'))\n\n\ndef _cargar_escala():\n    try:\n        with open(_RUTA_ESCALA, encoding=\'utf-8\') as f:\n            return json.load(f)[\'media_dn\']\n    except Exception:\n        return None\n\n\nESCALA_L4S = _cargar_escala()\n\n# ------------------------------------------------------------------- límites\nMAX_LADO_KM = float(_env(\'MAX_LADO_KM\', \'12\'))\nAVISO_LADO_KM = float(_env(\'AVISO_LADO_KM\', \'6\'))\nSOLAPE = float(_env(\'SOLAPE\', \'0.5\'))\nCOBERTURA_MIN = float(_env(\'COBERTURA_MIN\', \'0.30\'))\nMAX_COLA = int(_env(\'MAX_COLA\', \'4\'))\nMINUTOS_RESULTADO = int(_env(\'MINUTOS_RESULTADO\', \'30\'))\n\n# -------------------------------------------------------------------- ámbito\n#   \'bloquear\': rechaza con 403 lo que caiga fuera. Es lo que va en la entrega.\n#   \'avisar\'  : deja correr y marca el resultado como fuera del ámbito.\n#   \'libre\'   : sin límite, solo para pruebas.\nAMBITO = _env(\'AMBITO\', \'bloquear\')\nNOMBRE_AMBITO = \'provincia de La Convención, Cusco\'\nRUTA_LIMITE = _env(\'RUTA_LIMITE\', os.path.join(RAIZ, \'limite_laconvencion.geojson\'))\nBBOX_ESTUDIO = (-13.4489, -73.9805, -11.1988, -71.9730)   # sur, oeste, norte, este\nMIN_DENTRO = float(_env(\'MIN_DENTRO\', \'0.5\'))\n\n# --------------------------------------------------------------------- pruebas\n# A 1 no sale a la red: fabrica un área sintética con una cicatriz.\nMODO_FALSO = _env(\'MODO_FALSO\', \'0\') == \'1\'\nCENTRO_POR_DEFECTO = (-12.86, -72.69)    # Quillabamba\n'
ARCHIVOS['backend/nucleo/recetas.py'] = '"""\nRecetas de entrada de cada modelo.\n\nUn ONNX no dice qué espera en cada canal. Eso va en una receta JSON junto al\n.onnx, con el mismo nombre de base:\n\n    modelos/deeplabv3plus.onnx\n    modelos/deeplabv3plus.json\n\nLa receta fija los canales y su orden, la normalización con sus constantes, el\ntipo de salida y el umbral. La función `armar` construye el tensor a partir de\ncanales ya expresados en unidades de Landslide4Sense, y la usan por igual el\nbackend (con imágenes de cubo) y la verificación (con los h5 de L4S). Así lo que\nse mide contra Landslide4Sense es exactamente lo que corre en producción.\n"""\nimport json\nimport os\n\nimport numpy as np\n\nfrom . import config\n\n# Posición de cada canal en los h5 de Landslide4Sense (128 x 128 x 14).\nIDX_L4S = {\'B01\': 0, \'B02\': 1, \'B03\': 2, \'B04\': 3, \'B05\': 4, \'B06\': 5, \'B07\': 6,\n           \'B08\': 7, \'B09\': 8, \'B10\': 9, \'B11\': 10, \'B12\': 11,\n           \'pendiente\': 12, \'elevacion\': 13}\nINDICES = {\'NDVI\': (\'B08\', \'B04\'), \'NDWI\': (\'B03\', \'B08\')}\n# Nombres de Landslide4Sense para los canales de relieve: B13 = pendiente y\n# B14 = DEM. Una receta puede usar cualquiera de los dos nombres.\nALIAS = {\'B13\': \'pendiente\', \'B14\': \'elevacion\', \'B1\': \'B01\', \'B2\': \'B02\', \'B3\': \'B03\',\n         \'B4\': \'B04\', \'B5\': \'B05\', \'B6\': \'B06\', \'B7\': \'B07\', \'B8\': \'B08\', \'B9\': \'B09\'}\nTERRENO = (\'pendiente\', \'elevacion\')\nSALIDAS = (\'sigmoide\', \'softmax\')\n\n\nclass RecetaInvalida(ValueError):\n    pass\n\n\ndef normalizar_nombres(r):\n    """B13 -> pendiente, B14 -> elevacion, B1 -> B01... en canales y constantes."""\n    al = lambda c: ALIAS.get(c, c)\n    r[\'canales\'] = [al(c) for c in r[\'canales\']]\n    n = r.get(\'normalizacion\', {})\n    if \'canales\' in n:\n        n[\'canales\'] = [al(c) for c in n[\'canales\']]\n    if \'por_canal\' in n:\n        n[\'por_canal\'] = {al(c): v for c, v in n[\'por_canal\'].items()}\n    if \'escala_terreno\' in r:\n        r[\'escala_terreno\'] = {al(c): v for c, v in r[\'escala_terreno\'].items()}\n    if \'rango_entrenamiento\' in r:\n        r[\'rango_entrenamiento\'] = {al(c): v for c, v in r[\'rango_entrenamiento\'].items()}\n    return r\n\n\ndef leer(ruta_json):\n    with open(ruta_json, encoding=\'utf-8\') as f:\n        r = json.load(f)\n    normalizar_nombres(r)\n    validar(r, os.path.basename(ruta_json))\n    return r\n\n\ndef validar(r, origen=\'receta\'):\n    for campo in (\'nombre\', \'onnx\', \'canales\', \'lado_px\', \'normalizacion\', \'salida\',\n                  \'umbral\'):\n        if campo not in r:\n            raise RecetaInvalida(f\'{origen}: falta "{campo}"\')\n    for c in r[\'canales\']:\n        if c not in IDX_L4S and c not in INDICES:\n            raise RecetaInvalida(f\'{origen}: canal desconocido "{c}"\')\n    if r[\'salida\'] not in SALIDAS:\n        raise RecetaInvalida(f\'{origen}: salida "{r["salida"]}" no es {SALIDAS}\')\n    n = r[\'normalizacion\']\n    if n.get(\'metodo\') not in (\'percentil\', \'media_desv\', \'ninguna\'):\n        raise RecetaInvalida(f\'{origen}: normalización "{n.get("metodo")}" desconocida\')\n    if n[\'metodo\'] != \'ninguna\':\n        for c in n.get(\'canales\', []):\n            if c not in n.get(\'por_canal\', {}):\n                raise RecetaInvalida(f\'{origen}: faltan las constantes de "{c}"\')\n    for c, k in r.get(\'escala_terreno\', {}).items():\n        if c not in TERRENO or not float(k) > 0:\n            raise RecetaInvalida(f\'{origen}: escala_terreno["{c}"] no es válida\')\n    if not 0 < float(r[\'umbral\']) < 1:\n        raise RecetaInvalida(f\'{origen}: el umbral tiene que estar entre 0 y 1\')\n\n\ndef bandas_s2(r):\n    """Bandas de Sentinel-2 que hay que pedir para esta receta."""\n    s = set()\n    for c in r[\'canales\']:\n        if c in INDICES:\n            s.update(INDICES[c])\n        elif c.startswith(\'B\'):\n            s.add(c)\n    return sorted(s)\n\n\ndef necesita_dem(r):\n    return any(c in TERRENO for c in r[\'canales\'])\n\n\ndef _indice(a, b):\n    a = a.astype(np.float32); b = b.astype(np.float32)\n    den = a + b\n    out = np.zeros_like(a)\n    np.divide(a - b, den, out=out, where=den != 0)\n    return np.clip(out, -1.0, 1.0)\n\n\ndef armar(fuente, valido, r):\n    """fuente: canal -> (H, W) en unidades de Landslide4Sense.\n    valido: (H, W) bool. Devuelve (C, H, W) float32 listo para el ONNX."""\n    n = r[\'normalizacion\']\n    a_normalizar = set(n.get(\'canales\', []))\n    capas = []\n    for c in r[\'canales\']:\n        if c in INDICES:\n            x = _indice(fuente[INDICES[c][0]], fuente[INDICES[c][1]])\n        else:\n            x = fuente[c].astype(np.float32)\n        if c in a_normalizar:\n            k = n[\'por_canal\'][c]\n            if n[\'metodo\'] == \'percentil\':\n                lo, hi = float(k[\'lo\']), float(k[\'hi\'])\n                x = np.clip((x - lo) / max(hi - lo, 1e-9), 0.0, 1.0)\n            elif n[\'metodo\'] == \'media_desv\':\n                x = (x - float(k[\'media\'])) / max(float(k[\'desv\']), 1e-9)\n        capas.append(np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0))\n    t = np.stack(capas).astype(np.float32)\n    return t * valido.astype(np.float32)[None]\n\n\n# ------------------------------------------------------------ dos adaptadores\ndef desde_l4s(h5, r):\n    """Un parche de Landslide4Sense, (128, 128, 14) o (14, 128, 128), ya viene en\n    sus unidades. Solo hay que tomar los canales."""\n    a = np.asarray(h5, np.float32)\n    if a.shape[-1] == 14:\n        a = a.transpose(2, 0, 1)\n    valido = np.isfinite(a).all(axis=0)\n    a = np.nan_to_num(a, nan=0.0)\n    necesarios = set()\n    for c in r[\'canales\']:\n        necesarios.update(INDICES.get(c, (c,)))\n    fuente = {c: a[IDX_L4S[c]] for c in necesarios}\n    return armar(fuente, valido, r), valido\n\n\ndef desde_l2a(bandas_dn, terreno, valido, r, escala=None):\n    """Bandas de Sentinel-2 L2A en niveles digitales (ya sin el desplazamiento de\n    la baseline 04.00), y pendiente en grados y elevación en metros. Se llevan a\n    unidades de Landslide4Sense dividiendo cada banda por su media de escena,\n    que es como se construyó Landslide4Sense."""\n    escala = escala or r.get(\'escala_bandas\') or config.ESCALA_L4S\n    if not escala:\n        raise RecetaInvalida(\'no hay escala de bandas: falta escala_laconvencion.json\')\n    fuente = {}\n    for b, x in bandas_dn.items():\n        if b not in escala:\n            raise RecetaInvalida(f\'no hay constante de escala para {b}\')\n        fuente[b] = x.astype(np.float32) / float(escala[b])\n    # pendiente en grados y elevación en metros, a unidades de Landslide4Sense;\n    # si la receta no trae sus constantes valen las de Landslide4Sense\n    k_terreno = dict(config.ESCALA_TERRENO_L4S, **r.get(\'escala_terreno\', {}))\n    for c, x in (terreno or {}).items():\n        fuente[c] = x.astype(np.float32) / float(k_terreno[c])\n    return armar(fuente, valido, r)\n\n\ndef probabilidad(salida, tipo):\n    """(N, C, H, W) -> (N, H, W) en [0, 1] para la clase deslizamiento."""\n    z = np.asarray(salida, np.float32)\n    if tipo == \'sigmoide\':\n        if z.shape[1] != 1:\n            raise RecetaInvalida(f\'la receta dice sigmoide y el modelo da {z.shape[1]} canales\')\n        return 1.0 / (1.0 + np.exp(-np.clip(z[:, 0], -60, 60)))\n    if z.shape[1] != 2:\n        raise RecetaInvalida(f\'la receta dice softmax y el modelo da {z.shape[1]} canales\')\n    z = z - z.max(axis=1, keepdims=True)\n    e = np.exp(z)\n    return e[:, 1] / e.sum(axis=1)\n'
ARCHIVOS['backend/nucleo/modelos.py'] = '"""\nCarga de modelos ONNX con su receta, y la comprobación de que están entrenados.\n\nLos ONNX que había en el repositorio corrían sin error y no detectaban nada: sus\npesos eran los de inicialización de PyTorch, porque al exportar no se cargó el\ncheckpoint. Un fallo así no se ve en la forma del grafo ni en un error de\nejecución, solo en los pesos. Por eso cada modelo pasa por `revisar_pesos` antes\nde usarse: en una capa recién inicializada, el peso máximo es exactamente\n1/sqrt(fan_in), la media de |w| es la mitad y la desviación la de una uniforme.\nUna capa entrenada se aleja de eso.\n"""\nimport glob\nimport hashlib\nimport os\nimport threading\n\nimport numpy as np\n\nfrom . import config, recetas\n\nCABECERA_LFS = b\'version https://git-lfs\'\n_LOCK = threading.Lock()\n_SESIONES = {}\n_FICHAS = {}\n_RUTA_SESION = {}     # nombre -> ONNX con el que se abre la sesión\n\n\n# ------------------------------------------------------------------ archivos\ndef es_puntero_lfs(ruta):\n    try:\n        with open(ruta, \'rb\') as f:\n            return f.read(len(CABECERA_LFS)) == CABECERA_LFS\n    except OSError:\n        return False\n\n\ndef sha256(ruta):\n    h = hashlib.sha256()\n    with open(ruta, \'rb\') as f:\n        for bloque in iter(lambda: f.read(1 << 20), b\'\'):\n            h.update(bloque)\n    return h.hexdigest()\n\n\ndef _asegurar_archivo(ruta):\n    """Si falta o es un puntero de Git LFS, lo baja de URL_PESOS_BASE."""\n    if os.path.exists(ruta) and not es_puntero_lfs(ruta):\n        return\n    if not config.URL_PESOS_BASE:\n        raise FileNotFoundError(\n            f\'{os.path.basename(ruta)} falta o llegó como puntero de Git LFS, y no hay \'\n            \'URL_PESOS_BASE para bajarlo\')\n    import httpx\n    url = f\'{config.URL_PESOS_BASE.rstrip("/")}/{os.path.basename(ruta)}\'\n    print(f\'[pesos] bajando {url}\', flush=True)\n    tmp = ruta + \'.parcial\'\n    with httpx.stream(\'GET\', url, follow_redirects=True, timeout=600) as r:\n        r.raise_for_status()\n        with open(tmp, \'wb\') as f:\n            for trozo in r.iter_bytes(1 << 20):\n                f.write(trozo)\n    if es_puntero_lfs(tmp):\n        os.remove(tmp)\n        raise RuntimeError(f\'{url} devolvió otro puntero LFS, no el archivo\')\n    os.replace(tmp, ruta)\n\n\n# --------------------------------------------------------- prueba de los pesos\ndef _parece_inicializacion(w):\n    """¿Esta capa conv tiene la forma exacta de una inicialización y no de un\n    peso entrenado? Se reconocen las de PyTorch por defecto, Kaiming y Xavier,\n    uniformes o normales, que son las que usan torchvision y\n    segmentation_models_pytorch. Un peso entrenado tiene colas y no calza con\n    ninguna a la vez en forma y en escala."""\n    w = np.asarray(w, np.float64)\n    fi = w.shape[1] * w.shape[2] * w.shape[3]\n    fo = w.shape[0] * w.shape[2] * w.shape[3]\n    c = np.abs(w).max()\n    s = w.std()\n    if c == 0 or s == 0:\n        return False\n    cotas = (1 / np.sqrt(fi), np.sqrt(3 / fi), np.sqrt(6 / fi), np.sqrt(6 / (fi + fo)),\n             np.sqrt(3 / fo), np.sqrt(6 / fo))\n    uniforme = (0.47 < np.abs(w).mean() / c < 0.53 and 0.95 < s / (c / np.sqrt(3)) < 1.05\n                and any(abs(c / k - 1) < 0.03 for k in cotas))\n    if uniforme:\n        return True\n    z = (w - w.mean()) / s\n    exceso_curtosis = float((z ** 4).mean() - 3)\n    desvs = (np.sqrt(2 / fi), np.sqrt(2 / fo), np.sqrt(1 / fi), np.sqrt(2 / (fi + fo)))\n    normal = (abs(w.mean()) < 0.05 * s and abs(exceso_curtosis) < 0.15\n              and any(abs(s / k - 1) < 0.02 for k in desvs))\n    return bool(normal)\n\n\ndef _forma(valor):\n    return [d.dim_value if d.HasField(\'dim_value\') else \'N\'\n            for d in valor.type.tensor_type.shape.dim]\n\n\ndef revisar_pesos(ruta_onnx, copia_aparte=None):\n    """Cuántas capas conv tienen pesos indistinguibles de la inicialización\n    uniforme de PyTorch, U(-1/sqrt(fan_in), +1/sqrt(fan_in)). De paso devuelve\n    la forma de la entrada y de la salida, para no abrir una sesión solo para eso."""\n    import onnx\n    from onnx import numpy_helper\n    m = onnx.load(ruta_onnx, load_external_data=True)\n    pesos = {t.name: t for t in m.graph.initializer}\n    nombres_init = set(pesos)\n    entradas = [i for i in m.graph.input if i.name not in nombres_init]\n    forma_in = _forma(entradas[0]) if entradas else []\n    forma_out = _forma(m.graph.output[0]) if m.graph.output else []\n    marcas = []                      # en el orden del grafo\n    for nodo in m.graph.node:\n        if nodo.op_type != \'Conv\' or len(nodo.input) < 2 or nodo.input[1] not in pesos:\n            continue\n        t = pesos[nodo.input[1]]\n        if len(t.dims) != 4 or int(np.prod(t.dims)) < 256:\n            continue\n        marcas.append(_parece_inicializacion(numpy_helper.to_array(t)))\n    del pesos\n    ruta_sesion = _guardar_pesos_aparte(m, ruta_onnx, copia_aparte) if copia_aparte else ruta_onnx\n    del m\n    liberar_memoria()\n    convs, en_init = len(marcas), sum(marcas)\n    frac = en_init / convs if convs else 0.0\n    cabeza = any(marcas[-3:])\n    return {\'capas_conv\': convs, \'capas_en_inicializacion\': en_init,\n            \'fraccion\': round(frac, 3), \'cabeza_sin_entrenar\': cabeza,\n            # la cabeza es la que decide la clase: si está en la inicialización el\n            # modelo no aprendió a segmentar, aunque el codificador traiga ImageNet\n            \'sin_entrenar\': frac >= 0.25 or cabeza,\n            \'_forma_entrada\': forma_in, \'_forma_salida\': forma_out,\n            \'_ruta_sesion\': ruta_sesion}\n\n\ndef _guardar_pesos_aparte(m, ruta_onnx, destino):\n    """Copia del ONNX con los pesos en un .onnx.data al lado. Si algo falla se usa\n    el original: es una optimización de memoria, no un requisito."""\n    import onnx\n    try:\n        if os.path.getsize(ruta_onnx) < 8 * 1024 * 1024:\n            return ruta_onnx                  # modelos chicos: no vale la pena\n        os.makedirs(destino, exist_ok=True)\n        base = os.path.splitext(os.path.basename(ruta_onnx))[0]\n        salida = os.path.join(destino, base + \'.onnx\')\n        datos_ext = salida + \'.data\'\n        # onnx agrega al final de un .data existente en vez de reemplazarlo: sin\n        # borrarlo antes, cada arranque lo haría crecer otros 90 MB\n        for viejo in (salida, datos_ext):\n            if os.path.exists(viejo):\n                os.remove(viejo)\n        esperado = sum(len(t.raw_data) for t in m.graph.initializer)\n        onnx.save_model(m, salida, save_as_external_data=True, all_tensors_to_one_file=True,\n                        location=os.path.basename(datos_ext), size_threshold=1024)\n        if os.path.getsize(datos_ext) > 1.05 * esperado + 1024 * 1024:\n            raise RuntimeError(f\'la copia de pesos pesa {os.path.getsize(datos_ext)} bytes\')\n        return salida\n    except Exception as e:\n        print(f\'[modelos] sin copia con pesos aparte ({type(e).__name__}: {e}); \'\n              \'se usa el ONNX original\', flush=True)\n        return ruta_onnx\n\n\ndef liberar_memoria():\n    """Devuelve al sistema la memoria que Python ya soltó. Sin esto, glibc la\n    retiene y el proceso parece ocupar más de lo que usa."""\n    import gc\n    gc.collect()\n    try:\n        import ctypes\n        ctypes.CDLL(\'libc.so.6\').malloc_trim(0)\n    except Exception:\n        pass\n\n\n# ----------------------------------------------------------------- catálogo\ndef catalogo():\n    """nombre -> receta, leyendo modelos/*.json."""\n    out = {}\n    for ruta in sorted(glob.glob(os.path.join(config.CARPETA_MODELOS, \'*.json\'))):\n        try:\n            r = recetas.leer(ruta)\n        except Exception as e:\n            print(f\'[modelos] receta ignorada {os.path.basename(ruta)}: {e}\', flush=True)\n            continue\n        r[\'_ruta_onnx\'] = os.path.join(config.CARPETA_MODELOS, r[\'onnx\'])\n        out[r[\'nombre\']] = r\n    return out\n\n\ndef ficha(nombre):\n    """Lo que /modelos publica de cada modelo. Se calcula una vez."""\n    if nombre in _FICHAS:\n        return _FICHAS[nombre]\n    cat = catalogo()\n    if nombre not in cat:\n        raise KeyError(nombre)\n    r = cat[nombre]\n    f = {\'nombre\': nombre, \'etiqueta\': r.get(\'etiqueta\', nombre),\n         \'autor\': r.get(\'autor\'), \'canales\': r[\'canales\'], \'lado_px\': r[\'lado_px\'],\n         \'salida\': r[\'salida\'], \'umbral\': r[\'umbral\'],\n         \'necesita_dem\': recetas.necesita_dem(r),\n         \'entrenamiento\': r.get(\'entrenamiento\'),\n         \'verificacion_l4s\': r.get(\'verificacion_l4s\')}\n    try:\n        _asegurar_archivo(r[\'_ruta_onnx\'])\n        # La revisión carga el ONNX entero en memoria un momento; se hace antes de\n        # cualquier sesión para que los dos picos no se sumen.\n        pesos = revisar_pesos(r[\'_ruta_onnx\'], config.CARPETA_COPIAS or None)\n        _RUTA_SESION[nombre] = pesos.pop(\'_ruta_sesion\')\n        forma_in = pesos.pop(\'_forma_entrada\')\n        forma_out = pesos.pop(\'_forma_salida\')\n        f[\'entrada\'] = {\'forma\': forma_in}\n        f[\'salida_forma\'] = forma_out\n        problemas = []\n        if len(forma_in) != 4:\n            problemas.append(f\'el ONNX tiene una entrada de forma {forma_in}; se esperaba N×C×H×W\')\n        else:\n            c, h, w = forma_in[1], forma_in[2], forma_in[3]\n            if c != len(r[\'canales\']):\n                problemas.append(f\'el ONNX pide {c} canales y la receta da {len(r["canales"])}\')\n            if (h, w) != (r[\'lado_px\'], r[\'lado_px\']):\n                problemas.append(f\'el ONNX pide {h}x{w} y la receta dice {r["lado_px"]}\')\n        f[\'pesos\'] = pesos\n        f[\'sha256\'] = sha256(r[\'_ruta_onnx\'])\n        if pesos[\'sin_entrenar\']:\n            f[\'estado\'] = \'sin_entrenar\'\n            f[\'detalle\'] = (f\'{pesos["capas_en_inicializacion"]} de {pesos["capas_conv"]} \'\n                            \'capas tienen los pesos de inicialización: este ONNX se \'\n                            \'exportó sin cargar el entrenamiento y no detecta nada.\')\n        elif problemas:\n            f[\'estado\'] = \'error\'\n            f[\'detalle\'] = \'; \'.join(problemas)\n        else:\n            f[\'estado\'] = \'listo\'\n    except Exception as e:\n        f[\'estado\'] = \'error\'\n        f[\'detalle\'] = f\'{type(e).__name__}: {e}\'\n    _FICHAS[nombre] = f\n    return f\n\n\ndef _crear_sesion(ruta):\n    import onnxruntime as ort\n    so = ort.SessionOptions()\n    so.intra_op_num_threads = config.HILOS_ONNX\n    so.inter_op_num_threads = 1\n    so.enable_cpu_mem_arena = False\n    so.graph_optimization_level = {\n        \'basica\': ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,\n        \'extendida\': ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,\n        \'todas\': ort.GraphOptimizationLevel.ORT_ENABLE_ALL,\n    }.get(config.NIVEL_OPT_ONNX, ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED)\n    return ort.InferenceSession(ruta, so, providers=[\'CPUExecutionProvider\'])\n\n\ndef receta_lista(nombre):\n    """La receta de un modelo que puede usarse, sin cargar la sesión; o una\n    excepción que explica por qué no. Sirve para rechazar antes de bajar imágenes."""\n    cat = catalogo()\n    if nombre not in cat:\n        raise KeyError(f\'modelo "{nombre}" no existe. Disponibles: {sorted(cat)}\')\n    f = ficha(nombre)\n    if f[\'estado\'] == \'sin_entrenar\' and not config.PERMITIR_SIN_ENTRENAR:\n        raise PermissionError(f[\'detalle\'])\n    if f[\'estado\'] == \'error\':\n        raise RuntimeError(f[\'detalle\'])\n    return cat[nombre]\n\n\ndef obtener(nombre):\n    """(sesión, receta) lista para inferir, o una excepción que explica por qué no."""\n    receta = receta_lista(nombre)\n    with _LOCK:\n        if nombre not in _SESIONES:\n            if config.UN_MODELO_A_LA_VEZ and _SESIONES:\n                _SESIONES.clear()\n                liberar_memoria()\n            _SESIONES[nombre] = _crear_sesion(_RUTA_SESION.get(nombre, receta[\'_ruta_onnx\']))\n        return _SESIONES[nombre], receta\n\n\ndef soltar(nombre=None):\n    """Descarga la sesión (o todas) para que la memoria quede libre entre trabajos.\n    Volver a crearla cuesta menos de un segundo; tenerla cargada mientras se bajan\n    las imágenes cuesta más de 100 MB en un plan de 512."""\n    with _LOCK:\n        if nombre is None:\n            _SESIONES.clear()\n        else:\n            _SESIONES.pop(nombre, None)\n    liberar_memoria()\n\n\ndef cargados():\n    return sorted(_SESIONES)\n'
ARCHIVOS['herramientas/entrenar.py'] = '"""\nEntrena U-Net o DeepLabV3+ con Landslide4Sense, con las entradas que pide el\nproyecto: Sentinel-2 B1, B2, B3, B4, pendiente (B13) y DEM (B14).\n\nQué hace, en orden:\n\n  1. Lee los h5 de Landslide4Sense (128 x 128 x 14) de los tres conjuntos\n     oficiales: entrenamiento (3799), validación (245) y prueba (800).\n  2. Calcula la media y la desviación de cada canal sobre el entrenamiento.\n  3. Arma cada parche con `recetas.desde_l4s`, la MISMA función que usa el\n     servidor en Render. Entrenamiento y detección normalizan igual por\n     construcción, no por cuidado.\n  4. Entrena con BCE + Dice, AdamW, coseno y giros/espejos (como la Fase 4).\n     Guarda la época con mejor IoU en validación.\n  5. Elige el umbral que maximiza el IoU en validación y recién entonces mide\n     una sola vez en prueba.\n  6. Deja tres archivos con la misma firma, que exportar_onnx.py lee sin\n     opciones extra:\n        mejor_<arq>_b1b4b13b14_<firma>.pt\n        stats_norm_<firma>.json\n        resultados_<arq>_b1b4b13b14_<firma>.json\n\nUso (Colab con GPU):\n  python3 entrenar.py --l4s /content/l4s --arquitectura unet --salida /content/drive/MyDrive/entrenamiento\n  python3 entrenar.py --l4s /content/l4s --arquitectura deeplabv3plus --salida ...\n\nRequiere: torch, segmentation-models-pytorch, h5py, numpy.\n"""\nimport argparse\nimport csv\nimport glob\nimport hashlib\nimport json\nimport os\nimport re\nimport sys\nimport time\n\nimport numpy as np\n\nAQUI = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, os.path.join(AQUI, \'..\', \'backend\'))\n\nCANALES_PROYECTO = [\'B01\', \'B02\', \'B03\', \'B04\', \'pendiente\', \'elevacion\']\nETIQUETA_CANALES = \'b1b4b13b14\'\n\n\n# ------------------------------------------------------------------- datos\ndef pares(raiz, split, limite=None):\n    """(imagen, máscara) emparejados por nombre: images/<split>/image_12.h5 <->\n    annotations/<split>/mask_12.h5. Solo esos nombres: el espejo de Hugging Face\n    trae en annotations/train cinco archivos vacíos llamados image_*.h5 que no son\n    máscaras. Un par con algún archivo vacío o roto se salta y se avisa."""\n    import h5py\n    num = lambda f: int(re.search(r\'_(\\d+)\\.h5$\', os.path.basename(f)).group(1))\n    imgs = {num(f): f for f in glob.glob(os.path.join(raiz, \'images\', split, \'image_*.h5\'))}\n    msks = {num(f): f for f in glob.glob(os.path.join(raiz, \'annotations\', split, \'mask_*.h5\'))}\n    comunes = sorted(set(imgs) & set(msks))\n    if not comunes:\n        raise SystemExit(f\'no hay pares imagen/máscara en {raiz}/images/{split}\')\n    sin_par = len(set(imgs) ^ set(msks))\n    if sin_par:\n        print(f\'  aviso: {sin_par} archivo(s) de {split} sin su par; no se usan\')\n    if limite:\n        comunes = comunes[:limite]\n    buenos, malos = [], []\n    for k in comunes:\n        try:\n            with h5py.File(imgs[k], \'r\') as f:\n                ok_i = len(f.keys()) > 0 and f[list(f.keys())[0]].shape[-1] == 14\n            with h5py.File(msks[k], \'r\') as f:\n                ok_m = len(f.keys()) > 0 and f[list(f.keys())[0]].size == 128 * 128\n        except OSError:\n            ok_i = ok_m = False\n        (buenos if ok_i and ok_m else malos).append(k)\n    if malos:\n        print(f\'  aviso: {len(malos)} par(es) de {split} con archivos vacíos o rotos; se \'\n              f\'saltan: {[os.path.basename(imgs[k]) for k in malos[:10]]}\')\n    return [(imgs[k], msks[k]) for k in buenos]\n\n\ndef leer_h5(ruta):\n    import h5py\n    with h5py.File(ruta, \'r\') as f:\n        return np.asarray(f[list(f.keys())[0]])\n\n\ndef estadisticas(lista, canales):\n    """Media y desviación de cada canal sobre los píxeles válidos del entrenamiento,\n    en unidades de Landslide4Sense."""\n    from nucleo.recetas import IDX_L4S\n    idx = [IDX_L4S[c] for c in canales]\n    s = np.zeros(len(idx)); s2 = np.zeros(len(idx)); n = 0\n    muestras = [[] for _ in idx]\n    for img, _ in lista:\n        a = leer_h5(img).astype(np.float64)\n        if a.shape[-1] == 14:\n            a = a.transpose(2, 0, 1)\n        a = a[idx]\n        v = np.isfinite(a).all(axis=0)\n        x = a[:, v]\n        s += x.sum(axis=1); s2 += (x ** 2).sum(axis=1); n += x.shape[1]\n        for k in range(len(idx)):\n            muestras[k].append(x[k, ::37])\n    media = s / n\n    desv = np.sqrt(np.maximum(s2 / n - media ** 2, 1e-12))\n    # el rango que vio el modelo: el servidor avisa si un área cae fuera de él\n    rango = [np.percentile(np.concatenate(m), [0.1, 99.9]) for m in muestras]\n    return {c: {\'media\': round(float(m), 6), \'desv\': round(float(d), 6),\n                \'min\': round(float(r[0]), 6), \'max\': round(float(r[1]), 6)}\n            for c, m, d, r in zip(canales, media, desv, rango)}\n\n\ndef receta_de(canales, stats):\n    """La receta con la que se arma el tensor: la misma que irá junto al ONNX."""\n    return {\'nombre\': \'entrenamiento\', \'onnx\': \'-\', \'canales\': canales, \'lado_px\': 128,\n            \'salida\': \'sigmoide\', \'umbral\': 0.5,\n            \'normalizacion\': {\'metodo\': \'media_desv\',\n                              \'canales\': [c for c in canales if c in stats],\n                              \'por_canal\': stats}}\n\n\ndef cargar(lista, receta):\n    """Tensores (N, C, 128, 128) float32, máscaras (N, 128, 128) y válidos."""\n    from nucleo import recetas\n    X = np.zeros((len(lista), len(receta[\'canales\']), 128, 128), np.float32)\n    Y = np.zeros((len(lista), 128, 128), np.float32)\n    V = np.zeros((len(lista), 128, 128), np.float32)\n    for i, (img, msk) in enumerate(lista):\n        t, v = recetas.desde_l4s(leer_h5(img), receta)\n        X[i], V[i] = t, v\n        Y[i] = (np.asarray(leer_h5(msk)).squeeze() > 0)\n    return X, Y, V\n\n\n# ------------------------------------------------------------------ modelo\ndef perdida_mixta(logits, y, v, peso_dice=0.5):\n    import torch.nn.functional as F\n    import torch\n    bce = F.binary_cross_entropy_with_logits(logits, y, reduction=\'none\')\n    bce = (bce * v).sum() / v.sum().clamp(min=1.0)\n    p = torch.sigmoid(logits) * v\n    yt = y * v\n    dice = 1.0 - (2.0 * (p * yt).sum() + 1.0) / (p.sum() + yt.sum() + 1.0)\n    return (1.0 - peso_dice) * bce + peso_dice * dice\n\n\nclass Conteo:\n    """tp, fp, fn, tn globales (sobre píxeles) para varios umbrales a la vez."""\n    def __init__(self, umbrales):\n        self.u = np.asarray(umbrales, np.float32)\n        self.c = np.zeros((len(self.u), 4), np.int64)\n\n    def sumar(self, prob, y, v):\n        ok = v > 0\n        p, t = prob[ok], y[ok] > 0\n        for i, u in enumerate(self.u):\n            m = p >= u\n            self.c[i] += [np.sum(m & t), np.sum(m & ~t), np.sum(~m & t), np.sum(~m & ~t)]\n\n    def metricas(self, i):\n        tp, fp, fn, tn = (int(x) for x in self.c[i])\n        pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)\n        return {\'umbral\': round(float(self.u[i]), 2), \'iou\': round(tp / max(tp + fp + fn, 1), 4),\n                \'f1\': round(2 * pr * rc / max(pr + rc, 1e-9), 4), \'precision\': round(pr, 4),\n                \'recall\': round(rc, 4), \'tp\': tp, \'fp\': fp, \'fn\': fn, \'tn\': tn}\n\n\ndef evaluar(modelo, X, Y, V, umbrales, disp, lote=32):\n    import torch\n    modelo.eval(); cont = Conteo(umbrales)\n    with torch.no_grad():\n        for i in range(0, len(X), lote):\n            x = torch.from_numpy(X[i:i + lote]).to(disp)\n            prob = torch.sigmoid(modelo(x)[:, 0]).float().cpu().numpy()\n            cont.sumar(prob, Y[i:i + lote], V[i:i + lote])\n    return cont\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--l4s\', required=True, help=\'carpeta con images/ y annotations/\')\n    ap.add_argument(\'--arquitectura\', required=True, choices=[\'unet\', \'deeplabv3plus\'])\n    ap.add_argument(\'--salida\', required=True)\n    ap.add_argument(\'--canales\', default=\',\'.join(CANALES_PROYECTO))\n    ap.add_argument(\'--encoder\', default=\'resnet34\')\n    ap.add_argument(\'--encoder-weights\', dest=\'encoder_weights\', default=\'imagenet\',\n                    help="\'imagenet\' o \'ninguno\'")\n    ap.add_argument(\'--epocas\', type=int, default=100)\n    ap.add_argument(\'--paciencia\', type=int, default=30)\n    ap.add_argument(\'--lote\', type=int, default=16)\n    ap.add_argument(\'--lr\', type=float, default=3e-4)\n    ap.add_argument(\'--weight-decay\', dest=\'wd\', type=float, default=1e-4)\n    ap.add_argument(\'--peso-dice\', dest=\'peso_dice\', type=float, default=0.5)\n    ap.add_argument(\'--semilla\', type=int, default=42)\n    ap.add_argument(\'--limite\', type=int, help=\'solo para probar: parches por conjunto\')\n    a = ap.parse_args()\n\n    import torch\n    import segmentation_models_pytorch as smp\n    from nucleo.recetas import ALIAS, IDX_L4S\n\n    canales = [ALIAS.get(c.strip(), c.strip()) for c in a.canales.split(\',\') if c.strip()]\n    malos = [c for c in canales if c not in IDX_L4S]\n    if malos:\n        raise SystemExit(f\'canales que no están en Landslide4Sense: {malos}\')\n    torch.manual_seed(a.semilla); np.random.seed(a.semilla)\n    rng = np.random.default_rng(a.semilla)\n    disp = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    amp = disp == \'cuda\'\n    os.makedirs(a.salida, exist_ok=True)\n\n    config = {\'dataset\': \'Landslide4Sense (ibm-nasa-geospatial/Landslide4sense, conjuntos oficiales)\',\n              \'canales\': canales, \'normalizacion\': \'media_desv\', \'encoder\': a.encoder,\n              \'encoder_weights\': a.encoder_weights, \'tam_parche\': 128, \'lote\': a.lote,\n              \'epocas\': a.epocas, \'paciencia\': a.paciencia, \'lr\': a.lr, \'weight_decay\': a.wd,\n              \'peso_dice\': a.peso_dice, \'semilla\': a.semilla,\n              \'particion\': \'oficial: entrenamiento, validación (época y umbral), prueba (una vez)\',\n              \'aumentos\': \'giros de 90° y espejo horizontal\', \'limite\': a.limite}\n    firma = hashlib.sha1(json.dumps(dict({k: config[k] for k in config if k != \'epocas\'},\n                                         arquitectura=a.arquitectura),\n                                    sort_keys=True).encode()).hexdigest()[:12]\n    base = f\'{a.arquitectura}_{ETIQUETA_CANALES}_{firma}\'\n    print(f\'firma {firma} | {a.arquitectura} {a.encoder} | canales {canales} | {disp}\')\n\n    t0 = time.time()\n    tr, va, te = (pares(a.l4s, s, a.limite) for s in (\'train\', \'validation\', \'test\'))\n    print(f\'parches: entrenamiento {len(tr)}, validación {len(va)}, prueba {len(te)}\')\n    stats = estadisticas(tr, canales)\n    ruta_stats = os.path.join(a.salida, f\'stats_norm_{firma}.json\')\n    json.dump(stats, open(ruta_stats, \'w\', encoding=\'utf-8\'), indent=1)\n    print(\'media y desviación del entrenamiento:\', json.dumps(stats))\n    if \'elevacion\' in stats:\n        print(f"rango del DEM en el entrenamiento: {stats[\'elevacion\'][\'min\']*772.82:.0f} a "\n              f"{stats[\'elevacion\'][\'max\']*772.82:.0f} m; pendiente hasta "\n              f"{stats.get(\'pendiente\', {}).get(\'max\', 0)*20.292:.0f}°")\n    receta = receta_de(canales, stats)\n    Xtr, Ytr, Vtr = cargar(tr, receta)\n    Xva, Yva, Vva = cargar(va, receta)\n    Xte, Yte, Vte = cargar(te, receta)\n    print(f\'datos en memoria en {time.time()-t0:.0f} s; píxeles de deslizamiento en \'\n          f\'entrenamiento: {Ytr.mean()*100:.2f} %\')\n\n    pesos = None if a.encoder_weights in (\'ninguno\', \'none\', \'\') else a.encoder_weights\n    clase = {\'unet\': smp.Unet, \'deeplabv3plus\': smp.DeepLabV3Plus}[a.arquitectura]\n    modelo = clase(encoder_name=a.encoder, encoder_weights=pesos, in_channels=len(canales),\n                   classes=1).to(disp)\n    opt = torch.optim.AdamW(modelo.parameters(), lr=a.lr, weight_decay=a.wd)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=a.epocas)\n    escalador = torch.amp.GradScaler(\'cuda\', enabled=amp)\n    ruta_ckpt = os.path.join(a.salida, f\'mejor_{base}.pt\')\n    historial = []\n    mejor, mejor_ep, sin_mejora = -1.0, 0, 0\n    for ep in range(1, a.epocas + 1):\n        modelo.train(); orden = rng.permutation(len(Xtr)); perdidas = []\n        te0 = time.time()\n        for i in range(0, len(orden) - 1, a.lote):\n            j = orden[i:i + a.lote]\n            if len(j) < 2:\n                continue                         # BatchNorm necesita al menos 2\n            k = int(rng.integers(0, 4)); espejo = bool(rng.integers(0, 2))\n            x, y, v = (np.rot90(m[j], k, axes=(-2, -1)) for m in (Xtr, Ytr, Vtr))\n            if espejo:\n                x, y, v = x[..., ::-1], y[..., ::-1], v[..., ::-1]\n            x, y, v = (torch.from_numpy(np.ascontiguousarray(m)).to(disp) for m in (x, y, v))\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=\'cuda\', enabled=amp):\n                logits = modelo(x)[:, 0]\n            loss = perdida_mixta(logits.float(), y, v, a.peso_dice)\n            escalador.scale(loss).backward()\n            escalador.step(opt); escalador.update()\n            perdidas.append(float(loss))\n        sched.step()\n        m = evaluar(modelo, Xva, Yva, Vva, [0.5], disp).metricas(0)\n        historial.append({\'epoca\': ep, \'perdida\': round(float(np.mean(perdidas)), 5),\n                          \'lr\': opt.param_groups[0][\'lr\'], **{k: m[k] for k in (\'iou\', \'f1\', \'precision\', \'recall\')},\n                          \'segundos\': round(time.time() - te0, 1)})\n        marca = \'\'\n        if m[\'iou\'] > mejor:\n            mejor, mejor_ep, sin_mejora = m[\'iou\'], ep, 0\n            torch.save({\'modelo\': modelo.state_dict(), \'arquitectura\': a.arquitectura,\n                        \'epoca\': ep, \'iou_val\': m[\'iou\'], \'firma\': firma, \'config\': config},\n                       ruta_ckpt)\n            marca = \'  <- mejor, guardado\'\n        else:\n            sin_mejora += 1\n        print(f"época {ep:3d} | pérdida {np.mean(perdidas):.4f} | IoU val {m[\'iou\']:.4f} | "\n              f"F1 {m[\'f1\']:.4f} | {time.time()-te0:.0f} s{marca}", flush=True)\n        if sin_mejora >= a.paciencia:\n            print(f\'sin mejora en {a.paciencia} épocas: se detiene\'); break\n    with open(os.path.join(a.salida, f\'historial_{base}.csv\'), \'w\', newline=\'\') as f:\n        w = csv.DictWriter(f, fieldnames=list(historial[0])); w.writeheader(); w.writerows(historial)\n\n    # el mejor modelo, el umbral elegido en validación y una sola medición en prueba\n    ck = torch.load(ruta_ckpt, map_location=disp, weights_only=False)\n    modelo.load_state_dict(ck[\'modelo\'])\n    umbrales = np.round(np.arange(0.05, 0.96, 0.05), 2)\n    cv = evaluar(modelo, Xva, Yva, Vva, umbrales, disp)\n    barrido = [cv.metricas(i) for i in range(len(umbrales))]\n    elegido = max(barrido, key=lambda r: r[\'iou\'])\n    prueba = evaluar(modelo, Xte, Yte, Vte, [elegido[\'umbral\']], disp).metricas(0)\n    resultados = {\'firma\': firma, \'arquitectura\': a.arquitectura, \'mejor_epoca\': mejor_ep,\n                  \'umbral_operacion\': elegido[\'umbral\'], \'validacion\': elegido,\n                  \'prueba\': prueba, \'barrido_validacion\': barrido, \'config\': config,\n                  \'minutos\': round((time.time() - t0) / 60, 1)}\n    ruta_res = os.path.join(a.salida, f\'resultados_{base}.json\')\n    json.dump(resultados, open(ruta_res, \'w\', encoding=\'utf-8\'), indent=1, ensure_ascii=False)\n    print(f"\\nmejor época {mejor_ep}; umbral {elegido[\'umbral\']} (elegido en validación)")\n    print(f"validación: IoU {elegido[\'iou\']:.4f}  F1 {elegido[\'f1\']:.4f}")\n    print(f"prueba    : IoU {prueba[\'iou\']:.4f}  F1 {prueba[\'f1\']:.4f}  "\n          f"precisión {prueba[\'precision\']:.4f}  exhaustividad {prueba[\'recall\']:.4f}")\n    print(\'archivos:\', ruta_ckpt, ruta_stats, ruta_res, sep=\'\\n  \')\n\n\nif __name__ == \'__main__\':\n    main()\n'
ARCHIVOS['herramientas/exportar_onnx.py'] = '"""\nExporta un checkpoint de la Fase 4 a ONNX, con su receta, y comprueba que el\nONNX resultante está entrenado y da lo mismo que PyTorch.\n\nCorrige el fallo de los ONNX del repositorio, que se exportaron sin cargar el\nentrenamiento. Aquí la carga es estricta (cualquier clave que no calce es un\nerror, no un aviso), se comprueba que los pesos cambiaron al cargar, y el ONNX\npasa la misma prueba forense que hace el backend antes de usarlo.\n\nUso, en Colab o en local:\n\n  python3 exportar_onnx.py \\\n      --checkpoint mejor_deeplabv3plus_6ch_23c2a9ca2368.pt \\\n      --stats stats_norm_23c2a9ca2368.json \\\n      --resultados resultados_deeplabv3plus_6ch_23c2a9ca2368.json \\\n      --salida modelos/ --nombre deeplabv3plus \\\n      --autor "Huarcaya Pumacayo, V. N."\n\nPara la U-Net de Leonardo es lo mismo con sus tres archivos y --nombre unet.\nSi el checkpoint no trae la configuración de la Fase 4 (por ejemplo, un\nstate_dict suelto), se dice a mano. Para las entradas de la especificación\n(Sentinel-2 B1 a B4, pendiente B13 y DEM B14 de ALOS PALSAR) y salida de 2 clases:\n\n  python3 exportar_onnx.py --checkpoint unet.pt --salida modelos/ --nombre unet \\\n      --arquitectura unet --encoder resnet34 --lado 128 \\\n      --canales B01,B02,B03,B04,B13,B14 --clases 2 \\\n      --normalizacion media_desv --stats stats_unet.json\n\n--stats acepta {"0": {"lo":..,"hi":..}} (índices de L4S, como la Fase 4),\n{"B01": {"media":..,"desv":..}} o {"mean": [...], "std": [...]} en el orden de\n--canales. Con --normalizacion ninguna no hace falta.\nRequiere: torch, segmentation-models-pytorch, onnx, onnxruntime.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport sys\n\nimport numpy as np\n\nAQUI = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, os.path.join(AQUI, \'..\', \'backend\'))\n\nCANALES_6CH = [\'B02\', \'B03\', \'B04\', \'B08\', \'NDVI\', \'NDWI\']\nNOMBRES_14 = [\'B01\', \'B02\', \'B03\', \'B04\', \'B05\', \'B06\', \'B07\', \'B08\', \'B09\', \'B10\',\n              \'B11\', \'B12\', \'pendiente\', \'elevacion\']\nIDX_L4S = {n: i for i, n in enumerate(NOMBRES_14)}\nALIAS = {\'B13\': \'pendiente\', \'B14\': \'elevacion\', \'B1\': \'B01\', \'B2\': \'B02\', \'B3\': \'B03\',\n         \'B4\': \'B04\', \'B5\': \'B05\', \'B6\': \'B06\', \'B7\': \'B07\', \'B8\': \'B08\', \'B9\': \'B09\'}\n# Constantes de Landslide4Sense: pendiente = grados / 20.292, DEM = metros / 772.82\nESCALA_TERRENO_L4S = {\'pendiente\': 20.292, \'elevacion\': 772.82}\n\n\ndef construir(arquitectura, encoder, n_canales, n_clases=1):\n    import segmentation_models_pytorch as smp\n    clase = {\'deeplabv3plus\': smp.DeepLabV3Plus, \'unet\': smp.Unet}.get(arquitectura)\n    if clase is None:\n        raise SystemExit(f\'arquitectura "{arquitectura}" no reconocida\')\n    return clase(encoder_name=encoder, encoder_weights=None, in_channels=n_canales,\n                 classes=n_clases)\n\n\ndef estado_del_checkpoint(ck):\n    """El state_dict, venga en el formato de la Fase 4 o suelto."""\n    if isinstance(ck, dict):\n        for clave in (\'modelo\', \'model_state_dict\', \'state_dict\', \'model\'):\n            if clave in ck and isinstance(ck[clave], dict):\n                return ck[clave]\n        if ck and all(hasattr(v, \'shape\') for v in ck.values()):\n            return ck\n    raise SystemExit(\'no encuentro los pesos en el checkpoint (claves: \'\n                     f\'{list(ck)[:8] if isinstance(ck, dict) else type(ck).__name__})\')\n\n\ndef constantes(stats, canales, metodo):\n    """Constantes de normalización por canal desde los formatos de --stats."""\n    out = {}\n    for i, c in enumerate(canales):\n        if c not in IDX_L4S:\n            continue                        # índices como NDVI: no se normalizan\n        if metodo == \'percentil\':\n            k = stats.get(str(IDX_L4S[c])) or stats.get(c)\n            if not k or \'lo\' not in k:\n                raise SystemExit(f\'--stats no trae lo y hi de {c}\')\n            out[c] = {\'lo\': float(k[\'lo\']), \'hi\': float(k[\'hi\'])}\n        else:\n            if \'mean\' in stats and \'std\' in stats:\n                out[c] = {\'media\': float(stats[\'mean\'][i]), \'desv\': float(stats[\'std\'][i])}\n            else:\n                k = stats.get(c) or stats.get(str(IDX_L4S[c]))\n                if not k or \'media\' not in k:\n                    raise SystemExit(f\'--stats no trae media y desv de {c}\')\n                out[c] = {\'media\': float(k[\'media\']), \'desv\': float(k[\'desv\'])}\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--checkpoint\', required=True)\n    ap.add_argument(\'--stats\')\n    ap.add_argument(\'--resultados\')\n    ap.add_argument(\'--salida\', required=True)\n    ap.add_argument(\'--nombre\', required=True)\n    ap.add_argument(\'--etiqueta\')\n    ap.add_argument(\'--autor\')\n    ap.add_argument(\'--opset\', type=int, default=17)\n    ap.add_argument(\'--arquitectura\', choices=[\'unet\', \'deeplabv3plus\'])\n    ap.add_argument(\'--encoder\')\n    ap.add_argument(\'--lado\', type=int)\n    ap.add_argument(\'--canales\', help=\'p. ej. B01,B02,B03,B04,B13,B14\')\n    ap.add_argument(\'--clases\', type=int, choices=[1, 2],\n                    help=\'1: salida sigmoide; 2: softmax (fondo, deslizamiento)\')\n    ap.add_argument(\'--normalizacion\', choices=[\'percentil\', \'media_desv\', \'ninguna\'])\n    ap.add_argument(\'--umbral\', type=float)\n    a = ap.parse_args()\n\n    import torch\n    from nucleo import modelos as backend_modelos\n\n    ck = torch.load(a.checkpoint, map_location=\'cpu\', weights_only=False)\n    cfg = ck.get(\'config\', {}) if isinstance(ck, dict) else {}\n    arq = a.arquitectura or (ck.get(\'arquitectura\') if isinstance(ck, dict) else None)\n    if not arq:\n        raise SystemExit(\'el checkpoint no dice su arquitectura: usa --arquitectura\')\n    encoder = a.encoder or cfg.get(\'encoder\', \'resnet34\')\n    lado = int(a.lado or cfg.get(\'tam_parche\', 128))\n    if a.canales:\n        canales = [ALIAS.get(c.strip(), c.strip()) for c in a.canales.split(\',\') if c.strip()]\n    elif cfg.get(\'canales\'):            # checkpoints de entrenar.py\n        canales = [ALIAS.get(c, c) for c in cfg[\'canales\']]\n    elif cfg.get(\'config_canales\'):\n        canales = CANALES_6CH if cfg[\'config_canales\'] == \'6ch\' else NOMBRES_14\n    else:\n        raise SystemExit(\'el checkpoint no dice sus canales: usa --canales\')\n    desconocidos = [c for c in canales if c not in IDX_L4S and c not in (\'NDVI\', \'NDWI\')]\n    if desconocidos:\n        raise SystemExit(f\'canales desconocidos: {desconocidos}\')\n    n = len(canales)\n    estado = estado_del_checkpoint(ck)\n    if any(k.startswith(\'module.\') for k in estado):\n        estado = {k[len(\'module.\'):]: v for k, v in estado.items()}\n    cabeza = [v for k, v in estado.items() if k.startswith(\'segmentation_head\') and v.ndim == 4]\n    n_clases = a.clases or (int(cabeza[0].shape[0]) if cabeza else 1)\n    print(f\'checkpoint : {os.path.basename(a.checkpoint)}\')\n    print(f\'modelo     : {arq}, codificador {encoder}, {n} canales {canales}, {lado}x{lado}, \'\n          f\'{n_clases} clase(s)\')\n    if isinstance(ck, dict):\n        print(f\'época {ck.get("epoca")}, IoU de validación {ck.get("iou_val")}, firma {ck.get("firma")}\')\n\n    modelo = construir(arq, encoder, n, n_clases)\n    antes = {k: v.clone() for k, v in modelo.state_dict().items()}\n    modelo.load_state_dict(estado, strict=True)\n    # solo los pesos que se inicializan al azar delatan si la carga ocurrió; los de\n    # BatchNorm arrancan en 1 y 0 en cualquier modelo nuevo\n    aleatorios = [k for k, v in antes.items() if v.dtype.is_floating_point and v.ndim >= 2]\n    cambiadas = sum(1 for k in aleatorios if not torch.equal(modelo.state_dict()[k], antes[k]))\n    print(f\'carga estricta: {cambiadas} de {len(aleatorios)} pesos aleatorios reemplazados\')\n    if cambiadas < len(aleatorios):\n        raise SystemExit(\'el checkpoint no reemplazó todos los pesos: no se exporta\')\n    modelo.eval()\n\n    os.makedirs(a.salida, exist_ok=True)\n    ruta_onnx = os.path.join(a.salida, f\'{a.nombre}.onnx\')\n    x = torch.randn(2, n, lado, lado)\n    torch.onnx.export(modelo, x, ruta_onnx, input_names=[\'input\'], output_names=[\'output\'],\n                      dynamic_axes={\'input\': {0: \'N\'}, \'output\': {0: \'N\'}},\n                      opset_version=a.opset, do_constant_folding=True, dynamo=False)\n    print(f\'ONNX escrito: {ruta_onnx} ({os.path.getsize(ruta_onnx)/1e6:.1f} MB)\')\n\n    import onnxruntime as ort\n    ses = ort.InferenceSession(ruta_onnx, providers=[\'CPUExecutionProvider\'])\n    xs = torch.rand(4, n, lado, lado)\n    with torch.no_grad():\n        y_t = modelo(xs).numpy()\n    y_o = ses.run(None, {\'input\': xs.numpy()})[0]\n    dif = float(np.abs(y_t - y_o).max())\n    print(f\'paridad PyTorch / ONNX: diferencia máxima de logit {dif:.2e}\')\n    if dif > 1e-3:\n        raise SystemExit(\'el ONNX no reproduce a PyTorch\')\n\n    pesos = {k: v for k, v in backend_modelos.revisar_pesos(ruta_onnx).items()\n             if not k.startswith(\'_\')}\n    print(f\'prueba de pesos: {pesos}\')\n    if pesos[\'sin_entrenar\']:\n        raise SystemExit(\'el ONNX tiene pesos de inicialización: o el checkpoint no se cargó, \'\n                         \'o el modelo se entrenó tan pocos pasos que sus capas finales siguen \'\n                         \'como al inicio (pasa con 2 o 3 épocas de prueba, no con un \'\n                         \'entrenamiento completo)\')\n\n    metodo = a.normalizacion or cfg.get(\'normalizacion\') or \\\n        (\'percentil\' if cfg.get(\'config_canales\') else None)\n    if metodo is None:\n        raise SystemExit(\'di cómo se normalizó al entrenar: --normalizacion\')\n    if metodo != \'ninguna\' and not a.stats:\n        raise SystemExit(f\'--normalizacion {metodo} necesita --stats\')\n    stats = json.load(open(a.stats, encoding=\'utf-8\')) if a.stats else {}\n    por_canal = constantes(stats, canales, metodo) if metodo != \'ninguna\' else {}\n    res = json.load(open(a.resultados, encoding=\'utf-8\')) if a.resultados else {}\n    terreno = [c for c in canales if c in ESCALA_TERRENO_L4S]\n    receta = {\n        \'nombre\': a.nombre,\n        \'etiqueta\': a.etiqueta or {\'deeplabv3plus\': \'DeepLabV3+\', \'unet\': \'U-Net\'}[arq]\n                    + f\' · {encoder}\',\n        \'autor\': a.autor,\n        \'onnx\': os.path.basename(ruta_onnx),\n        \'canales\': canales, \'lado_px\': lado,\n        \'normalizacion\': {\n            \'metodo\': metodo, \'percentiles\': cfg.get(\'percentiles\', [2, 98]),\n            \'canales\': list(por_canal),\n            \'por_canal\': por_canal,\n            \'nota\': \'constantes en unidades de Landslide4Sense; los índices NDVI y NDWI \'\n                    \'se calculan sobre esas unidades y no se normalizan\'},\n        \'salida\': \'sigmoide\' if n_clases == 1 else \'softmax\',\n        \'umbral\': float(a.umbral or res.get(\'umbral_operacion\', cfg.get(\'umbral_entren\', 0.5))),\n        \'entrenamiento\': {\n            \'firma\': ck.get(\'firma\') if isinstance(ck, dict) else None,\n            \'arquitectura\': arq, \'codificador\': encoder,\n            \'dataset\': cfg.get(\'dataset\', \'Landslide4Sense\'),\n            \'epoca\': ck.get(\'epoca\') if isinstance(ck, dict) else None,\n            \'iou_validacion\': res.get(\'validacion\', {}).get(\'iou\'),\n            \'prueba\': {k: res.get(\'prueba\', {}).get(k)\n                       for k in (\'iou\', \'f1\', \'precision\', \'recall\')},\n            \'particion\': cfg.get(\'particion\') or\n                         (\'grupos espaciales de TrainData (Fase 4)\' if cfg else None)},\n        **({\'rango_entrenamiento\': {c: {\'min\': stats[c][\'min\'], \'max\': stats[c][\'max\']}\n                                    for c in canales if isinstance(stats.get(c), dict)\n                                    and \'min\' in stats[c]}}\n           if any(isinstance(stats.get(c), dict) and \'min\' in stats[c] for c in canales) else {}),\n        **({\'escala_terreno\': {c: ESCALA_TERRENO_L4S[c] for c in terreno},\n            \'terreno_entrenamiento\': \'ALOS PALSAR (ASF) en Landslide4Sense: B13 pendiente, \'\n                                     \'B14 DEM\'} if terreno else {}),\n        \'exportacion\': {\'opset\': a.opset, \'paridad_max_logit\': dif, \'pesos\': pesos,\n                        \'sha256_onnx\': hashlib.sha256(open(ruta_onnx, \'rb\').read()).hexdigest()},\n    }\n    ruta_json = os.path.join(a.salida, f\'{a.nombre}.json\')\n    json.dump(receta, open(ruta_json, \'w\', encoding=\'utf-8\'), indent=1, ensure_ascii=False)\n    print(f\'receta escrita: {ruta_json}\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
ARCHIVOS['herramientas/verificar_onnx.py'] = '"""\nVerifica un ONNX con su receta sobre parches de Landslide4Sense con máscara.\n\nUsa la misma función que el backend (`recetas.desde_l4s` -> `recetas.armar`),\nasí que lo que se mide aquí es lo que corre en producción, no una copia.\n\n  python3 verificar_onnx.py --modelos ../backend/modelos --l4s /ruta/l4s \\\n      [--splits validation test] [--lista split_grupos.csv --grupo test] \\\n      [--escribir]\n\n--l4s apunta a la estructura del dataset de Hugging Face\n(ibm-nasa-geospatial/Landslide4sense): images/<split>/image_N.h5 y\nannotations/<split>/mask_N.h5.\n--escribir guarda el resultado en la receta (campo verificacion_l4s), que el\nbackend publica en /modelos y en el manifiesto de cada ZIP.\n"""\nimport argparse\nimport datetime as dt\nimport glob\nimport json\nimport os\nimport sys\n\nimport numpy as np\n\nAQUI = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, os.path.join(AQUI, \'..\', \'backend\'))\n\n\ndef pares(raiz, split, nombres=None):\n    imgs = glob.glob(os.path.join(raiz, \'images\', split, \'image_*.h5\'))\n    imgs.sort(key=lambda p: int(p.rsplit(\'_\', 1)[1][:-3]))\n    for p in imgs:\n        if nombres is not None and os.path.basename(p) not in nombres:\n            continue\n        q = p.replace(os.sep + \'images\' + os.sep, os.sep + \'annotations\' + os.sep) \\\n             .replace(\'image_\', \'mask_\')\n        if os.path.exists(q):\n            yield p, q\n\n\ndef evaluar(ses, receta, lista, lote=16):\n    import h5py\n    from nucleo import recetas\n    tp = fp = fn = tn = 0\n    ious = []\n    nombre = ses.get_inputs()[0].name\n    for i in range(0, len(lista), lote):\n        X, Y, V = [], [], []\n        for p, q in lista[i:i + lote]:\n            with h5py.File(p, \'r\') as f:\n                t, v = recetas.desde_l4s(f[\'img\'][:], receta)\n            with h5py.File(q, \'r\') as f:\n                y = np.asarray(f[\'mask\'][:]) > 0\n            X.append(t); Y.append(y & v); V.append(v)\n        prob = recetas.probabilidad(ses.run(None, {nombre: np.stack(X)})[0], receta[\'salida\'])\n        for pk, y, v in zip(prob, Y, V):\n            m = (pk >= receta[\'umbral\']) & v\n            a, b, c = int((m & y).sum()), int((m & ~y).sum()), int((~m & y & v).sum())\n            tp += a; fp += b; fn += c; tn += int((~m & ~y & v).sum())\n            if y.any():\n                ious.append(a / max(a + b + c, 1))\n    iou = tp / max(tp + fp + fn, 1)\n    pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)\n    return {\'parches\': len(lista), \'iou\': round(iou, 4),\n            \'f1\': round(2 * pr * rc / max(pr + rc, 1e-9), 4),\n            \'precision\': round(pr, 4), \'recall\': round(rc, 4),\n            \'iou_medio_parches_con_deslizamiento\': round(float(np.mean(ious)), 4) if ious else None,\n            \'tp\': tp, \'fp\': fp, \'fn\': fn, \'tn\': tn}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--modelos\', required=True)\n    ap.add_argument(\'--l4s\', required=True)\n    ap.add_argument(\'--splits\', nargs=\'+\', default=[\'validation\', \'test\'])\n    ap.add_argument(\'--lista\', help=\'CSV de la Fase 4 con columnas de archivo y split\')\n    ap.add_argument(\'--grupo\', default=\'test\')\n    ap.add_argument(\'--escribir\', action=\'store_true\')\n    a = ap.parse_args()\n\n    import onnxruntime as ort\n    from nucleo import recetas\n    from nucleo.modelos import revisar_pesos\n\n    for ruta_r in sorted(glob.glob(os.path.join(a.modelos, \'*.json\'))):\n        r = recetas.leer(ruta_r)\n        onnx = os.path.join(a.modelos, r[\'onnx\'])\n        print(f\'\\n===== {r["nombre"]} ({os.path.basename(onnx)})\')\n        pesos = {k: v for k, v in revisar_pesos(onnx).items() if not k.startswith(\'_\')}\n        print(f\'  pesos: {pesos}\')\n        if pesos[\'sin_entrenar\']:\n            print(\'  SIN ENTRENAR: no tiene sentido medirlo\'); continue\n        so = ort.SessionOptions(); so.intra_op_num_threads = os.cpu_count() or 2\n        ses = ort.InferenceSession(onnx, so, providers=[\'CPUExecutionProvider\'])\n        resultado = {\'fecha\': dt.date.today().isoformat(), \'umbral\': r[\'umbral\'],\n                     \'dataset\': \'ibm-nasa-geospatial/Landslide4sense (Hugging Face)\'}\n        if a.lista:\n            import pandas as pd\n            t = pd.read_csv(a.lista)\n            col_split = \'split\' if \'split\' in t.columns else t.columns[-1]\n            col_img = next(c for c in t.columns if \'img\' in c.lower() or \'imag\' in c.lower())\n            sel = t[t[col_split] == a.grupo]\n            nombres = {os.path.basename(str(x)) for x in sel[col_img]}\n            lista = list(pares(a.l4s, \'train\', nombres))\n            m = evaluar(ses, r, lista)\n            print(f\'  {a.grupo} de la Fase 4 ({len(lista)} de {len(nombres)} parches): {m}\')\n            resultado[f\'fase4_{a.grupo}\'] = m\n        for s in a.splits:\n            lista = list(pares(a.l4s, s))\n            if not lista:\n                continue\n            m = evaluar(ses, r, lista)\n            print(f\'  {s}: {m}\')\n            resultado[s] = m\n        ref = (r.get(\'entrenamiento\') or {}).get(\'prueba\', {})\n        if ref.get(\'iou\') is not None:\n            print(f\'  referencia del entrenamiento, prueba: IoU {ref["iou"]:.4f}, F1 {ref["f1"]:.4f}\')\n            medidos = [resultado[s][\'iou\'] for s in a.splits if s in resultado]\n            if medidos and max(medidos) < 0.6 * ref[\'iou\']:\n                print(\'  ATENCIÓN: el IoU medido queda muy por debajo del de entrenamiento. \'\n                      \'Revisa que el checkpoint, las estadísticas y la receta sean de la misma \'\n                      \'firma; con un ONNX bien exportado la diferencia es pequeña.\')\n                resultado[\'alerta\'] = \'IoU muy por debajo del de entrenamiento\'\n        if a.escribir:\n            crudo = json.load(open(ruta_r, encoding=\'utf-8\'))\n            crudo[\'verificacion_l4s\'] = resultado\n            json.dump(crudo, open(ruta_r, \'w\', encoding=\'utf-8\'), indent=1, ensure_ascii=False)\n            print(f\'  escrito en {os.path.basename(ruta_r)}\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
for r, c in ARCHIVOS.items():
    p = os.path.join('/content/artefacto', r)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, 'w', encoding='utf-8').write(c)
print('escritos:', list(ARCHIVOS))

escritos: ['backend/nucleo/__init__.py', 'backend/nucleo/config.py', 'backend/nucleo/recetas.py', 'backend/nucleo/modelos.py', 'herramientas/entrenar.py', 'herramientas/exportar_onnx.py', 'herramientas/verificar_onnx.py']


## Landslide4Sense

Los tres conjuntos oficiales con sus máscaras, del espejo de Hugging Face (~9 GB).

In [5]:
from huggingface_hub import snapshot_download
snapshot_download('ibm-nasa-geospatial/Landslide4sense', repo_type='dataset',
                  local_dir='/content/l4s', max_workers=16)
for d in ('images/train', 'annotations/train', 'images/validation', 'annotations/validation',
          'images/test', 'annotations/test'):
    print(d, len(os.listdir(f'/content/l4s/{d}')))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9695 files:   0%|          | 0/9695 [00:00<?, ?it/s]

images/train 3799
annotations/train 3804
images/validation 245
annotations/validation 245
images/test 800
annotations/test 800


In [6]:
import subprocess, sys
def correr(*args):
    """Corre un script mostrando su salida en vivo; si falla, el cuaderno se detiene."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    cola = []
    for linea in p.stdout:
        cola = (cola + [linea])[-60:]
        if 'Warning' not in linea and 'warn(' not in linea:
            print(linea, end='')
    if p.wait():
        print(''.join(cola[-30:]))
        raise RuntimeError('falló; revisa el mensaje de arriba')

## Entrenar

Cada modelo guarda en Drive su mejor época (por IoU de validación), las estadísticas
de normalización y los resultados con el umbral elegido en validación y la medición
en prueba. Si Colab se corta, lo ya guardado queda en Drive.

In [7]:
for arq in MODELOS:
    print('=' * 70, '\n', arq, '\n' + '=' * 70)
    correr('/content/artefacto/herramientas/entrenar.py', '--l4s', '/content/l4s',
           '--arquitectura', arq, '--canales', CANALES, '--salida', CARPETA_DRIVE,
           '--epocas', str(EPOCAS), '--paciencia', str(PACIENCIA))

 unet 
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  perdidas.append(float(loss))
firma c4a0d3781a67 | unet resnet34 | canales ['B01', 'B02', 'B03', 'B04', 'pendiente', 'elevacion'] | cuda
parches: entrenamiento 3799, validación 245, prueba 800
media y desviación del entrenamiento: {"B01": {"media": 0.925704, "desv": 0.140988, "min": 0.678172, "max": 1.800664}, "B02": {"media": 0.922701, "desv": 0.220698, "min": 0.602693, "max": 2.523551}, "B03": {"media": 0.954109, "desv": 0.318425, "min": 0.450697, "max": 3.12172}, "B04": {"media": 0.959639, "desv": 0.572375, "min": 0.354918, "max": 4.937407}, "pendiente": {"media": 1.251108, "desv": 0.678369, "min": 0.0, "max": 3.301748}, "elevacion": {"media": 1.649543, "desv": 1.072711, "min": 0.029761, "max": 4.737126}}
rango del DEM en el entrenamiento: 23 a 3661 m; pendiente hasta 67°
datos en memoria en 111 s; píxeles de deslizamiento en entrenamiento: 

## Exportar a ONNX y verificar

El exportador lee del checkpoint los canales y la normalización. Se detiene si la
carga no es estricta, si el ONNX no reproduce a PyTorch o si los pesos parecen sin
entrenar. Después la verificación mide el ONNX con la receta del servidor sobre
validación y prueba: **tiene que dar lo mismo que el entrenamiento**.

In [8]:
import glob, json, shutil
os.makedirs('/content/onnx', exist_ok=True)
for arq, autor in MODELOS.items():
    ck = sorted(glob.glob(f'{CARPETA_DRIVE}/mejor_{arq}_b1b4b13b14_*.pt'), key=os.path.getmtime)[-1]
    firma = os.path.splitext(ck)[0].rsplit('_', 1)[-1]
    correr('/content/artefacto/herramientas/exportar_onnx.py', '--checkpoint', ck,
           '--stats', f'{CARPETA_DRIVE}/stats_norm_{firma}.json',
           '--resultados', f'{CARPETA_DRIVE}/resultados_{arq}_b1b4b13b14_{firma}.json',
           '--salida', '/content/onnx', '--nombre', arq, '--autor', autor)
correr('/content/artefacto/herramientas/verificar_onnx.py', '--modelos', '/content/onnx',
       '--l4s', '/content/l4s', '--splits', 'validation', 'test', '--escribir')

  torch.onnx.export(modelo, x, ruta_onnx, input_names=['input'], output_names=['output'],
checkpoint : mejor_unet_b1b4b13b14_c4a0d3781a67.pt
modelo     : unet, codificador resnet34, 6 canales ['B01', 'B02', 'B03', 'B04', 'pendiente', 'elevacion'], 128x128, 1 clase(s)
época 16, IoU de validación 0.4637, firma c4a0d3781a67
carga estricta: 47 de 47 pesos aleatorios reemplazados
ONNX escrito: /content/onnx/unet.onnx (97.8 MB)
paridad PyTorch / ONNX: diferencia máxima de logit 5.72e-06
prueba de pesos: {'capas_conv': 46, 'capas_en_inicializacion': 0, 'fraccion': 0.0, 'cabeza_sin_entrenar': False, 'sin_entrenar': False}
receta escrita: /content/onnx/unet.json
  torch.onnx.export(modelo, x, ruta_onnx, input_names=['input'], output_names=['output'],
checkpoint : mejor_deeplabv3plus_b1b4b13b14_e838f4a37bf6.pt
modelo     : deeplabv3plus, codificador resnet34, 6 canales ['B01', 'B02', 'B03', 'B04', 'pendiente', 'elevacion'], 128x128, 1 clase(s)
época 62, IoU de validación 0.4616, firma e838f4a37b

In [9]:
print(f"{'modelo':15s} {'umbral':>6s} {'IoU val':>8s} {'IoU prueba':>10s} {'F1 prueba':>9s} {'ONNX = entrenamiento':>22s}")
for arq in MODELOS:
    r = json.load(open(f'/content/onnx/{arq}.json', encoding='utf-8'))
    ent, ver = r['entrenamiento']['prueba'], (r.get('verificacion_l4s') or {}).get('test', {})
    igual = ent.get('iou') is not None and abs(ent['iou'] - ver.get('iou', -1)) < 0.002
    print(f"{arq:15s} {r['umbral']:6.2f} {r['entrenamiento']['iou_validacion']:8.4f} "
          f"{ent['iou']:10.4f} {ent['f1']:9.4f} {'sí' if igual else 'NO: revisar':>22s}")
for f in os.listdir('/content/onnx'):
    shutil.copy(os.path.join('/content/onnx', f), SALIDA_ONNX)
print('\nguardado en', SALIDA_ONNX, sorted(os.listdir(SALIDA_ONNX)))

modelo          umbral  IoU val IoU prueba F1 prueba   ONNX = entrenamiento
unet              0.70   0.4656     0.4245    0.5960                     sí
deeplabv3plus     0.10   0.4810     0.4338    0.6051                     sí

guardado en /content/drive/MyDrive/onnx_para_render ['deeplabv3plus.json', 'deeplabv3plus.onnx', 'unet.json', 'unet.onnx']


## Siguiente paso

Lleva `unet.onnx`, `unet.json`, `deeplabv3plus.onnx` y `deeplabv3plus.json` a
`modelos/` del repositorio que lee Render y haz `git push`. En `/modelos` del
servidor los dos tienen que salir con estado `listo`.

Para la tesis, reporta las métricas de **prueba** (no las de validación, que se
usaron para elegir la época y el umbral) y di que son de Landslide4Sense, no de
La Convención.